In [ ]:
# Install necessary libraries
%pip install --quiet --upgrade google-cloud-vectorsearch fsspec gcsfs google-auth google-api-core scikit-learn

try:
    # Import libraries
    import google.auth
    import pandas as pd
    from datetime import datetime
    from google.cloud import vectorsearch_v1
    from google.cloud import vectorsearch_v1beta
    import time
    from sklearn.manifold import TSNE  # 라이브러리 바이너리 매핑 호환성 체크용 임포트

    print("라이브러리가 성공적으로 임포트되었습니다.")
except ImportError as e:
    print(f"⚠️ 패키지 업데이트 감지 (바이너리 불일치): {e}")
    print("새로운 라이브러리를 메모리에 반영하기 위해 주피터 커널을 자동으로 재시작합니다...")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)


In [ ]:
_, PROJECT_ID = google.auth.default()
LOCATION="asia-northeast1"
COLLECTION_ID="amazon-product-768-compact"

In [ ]:
df_golden_product = pd.read_parquet('gs://jk-amazon-products-index/golden-records/amazon_products_golden.parquet')
df_golden_product

In [ ]:
product_filter = [
    # 가전 / 디지털 / IT
    'CELLULAR_PHONE', 'NOTEBOOK_COMPUTER', 
    'PERSONAL_COMPUTER', 'MONITOR', 'KEYBOARDS', 'INPUT_MOUSE', 
    'HEADPHONES', 'SPEAKERS', 'COMPUTER_SPEAKER', 'MICROPHONE', 'TELEVISION', 
    'REMOTE_CONTROL', 'BATTERY', 'POWER_BANK', 'CHARGING_ADAPTER', 'FLASH_DRIVE', 
    'WEIGH_SCALE', 'CLOCK', 'SECURITY_CAMERA',
    
    # 주방 / 식기    
    'KITCHEN_KNIFE', 'SCISSORS', 'CAN_OPENER', 'BOTTLE_OPENER', 
    'DINNERWARE', 'DISHWARE_PLATE', 'DISHWARE_BOWL', 'DRINKING_CUP', 
    'FLATWARE', 'DRINKING_STRAW', 'DRINK_COASTER', 'PITCHER', 'THERMOS', 
    'PLACEMAT', 'POT_HOLDER', 'BOTTLE_RACK', 'DRYING_RACK', 'FOOD_STORAGE_BAG',

    # 뷰티 / 생활용품 / 위생
    'HAIR_BRUSH', 'HAIR_COMB', 'HAIR_IRON', 
    'SKIN_CLEANING_AGENT', 'SKIN_MOISTURIZER', 'SKIN_EXFOLIANT', 'SKIN_TREATMENT_MASK', 
    'SUNSCREEN', 'LIP_BALM', 'TOOTHBRUSH', 'TOOTH_CLEANING_AGENT', 
    'MOUTHWASH', 'FACIAL_TISSUE', 'SKIN_CLEANING_WIPE', 'CLEANING_AGENT', 
    'CLEANING_BRUSH', 'BROOM', 'MASCARA', 'NAIL_POLISH',

    # 식품 / 음료 / 식재료
    'WATER', 'COFFEE', 'TEA', 'JUICE_AND_JUICE_DRINK', 'MILK_SUBSTITUTE', 
    'BREAD', 'COOKIE', 'CAKE', 'PASTRY', 'CRACKER', 'PRETZEL', 
    'POPCORN', 'CANDY', 'CHOCOLATE_CANDY', 'SUGAR_CANDY', 'SNACK_CHIP_AND_CRISP', 
    'SNACK_FOOD_BAR', 'BREAKFAST_CEREAL', 'NOODLE', 'RICE_MIX', 'PACKAGED_SOUP_AND_STEW', 
    'MEAT', 'POULTRY', 'FISH', 'SHELLFISH', 'VEGETABLE', 'FRUIT', 
    'NUTS', 'LEGUME', 'FLOUR', 'SUGAR', 'SUGAR_SUBSTITUTE', 'HONEY', 
    'EDIBLE_OIL_VEGETABLE', 'DAIRY_BASED_BUTTER', 'DAIRY_BASED_CHEESE', 'DAIRY_BASED_CREAM', 
    'SAUCE', 'SALAD_DRESSING', 'VINEGAR', 'WINE',

    # 가구 / 인테리어 / 수납
    'BLANKET', 'SOFA', 'CHAIR', 'DESK', 'TABLE', 'BENCH',  
    'CABINET', 'SHELF', 'CURTAIN', 'HOME_MIRROR', 
    'VASE', 'TRASH_CAN', 'STORAGE_BOX', 'STORAGE_BAG',

    # 패션잡화 / 취미 / 반려용품
    'SHOES', 'SANDAL', 'BOOT', 'BAG', 'BACKPACK', 
    'HANDBAG', 'TOTE_BAG', 'WALLET', 'WATCH', 'SUNGLASSES', 'EARRING', 'NECKLACE', 'BRACELET', 
    'RING', 

    # 문구 / 공구 / 건강 / 기타
    'WRITING_INSTRUMENT', 'STAPLER', 'SELF_STICK_NOTE', 'LOCK', 'VITAMIN', 
    'DIETARY_SUPPLEMENTS', 'FIRST_AID_KIT']
df_compact_products = df_golden_product[df_golden_product['product_type'].isin(product_filter)]
df_compact_products

> [!NOTE]
> 아래 이후의 셀들은 Session 1 시작하기 전에 실행했던 `install.sh`가 수행했던 작업과 같은 내용으로 참고용도입니다. `install.sh`가 잘 실행되었다면 실행하실 필요는 없으며, 실행하셨더라도 `409 AlreadyExists`와 같은 에러가 나는 것이 정상입니다.

In [ ]:
!gcloud storage buckets create gs://$PROJECT_ID-vs2 --location=$LOCATION

In [ ]:
# 원하는 컬럼('id', 'vectors')만 선택하여 JSONL로 저장
df_compact_products[['id', 'data', 'vectors']].to_json(f'gs://{PROJECT_ID}-vs2/data/amazon-product-dataset-768-compact.jsonl', orient='records', lines=True, force_ascii=False)

In [ ]:
# Create the client
vector_search_service_client = vectorsearch_v1beta.VectorSearchServiceClient()

# The JSON schema for the data
data_schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "description": {"type": "string"},
    },
}

# The JSON schema for the vector
vector_schema = {
    "image_embedding": {"dense_vector": {"dimensions": 768}},
    "text_embedding": {"dense_vector": {"dimensions": 768}}
}

collection = vectorsearch_v1beta.Collection(
    data_schema=data_schema,
    vector_schema=vector_schema,
)
request = vectorsearch_v1beta.CreateCollectionRequest(
    parent=f"projects/{PROJECT_ID}/locations/{LOCATION}",
    collection_id=COLLECTION_ID,
    collection=collection,
)

# Create the collection
operation = vector_search_service_client.create_collection(request=request)

# Wait for the result (note this may take up to several minutes)
operation.result()

In [ ]:
# Create the client
vector_search_service_client = vectorsearch_v1.VectorSearchServiceClient()

# Initialize request
request = vectorsearch_v1.ImportDataObjectsRequest(
    name=f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/{COLLECTION_ID}",
    gcs_import={
      "contents_uri": f"gs://{PROJECT_ID}-vs2/data/",
      "error_uri": f"gs://{PROJECT_ID}-vs2/error/",
    },
)

# Make the request
print(datetime.now()) 
operation = vector_search_service_client.import_data_objects(request=request)

In [ ]:
while operation.done() == False:
    time.sleep(1)
print(datetime.now())
time.sleep(30)

In [ ]:
def create_index(index_field: str):
    # Create a client
    client = vectorsearch_v1beta.VectorSearchServiceClient()

    # Initialize request argument(s)
    index = vectorsearch_v1beta.Index(
        index_field=index_field,
        #filter_fields=["year", "genre"],
        store_fields=["name", "description"],
    )
    request = vectorsearch_v1beta.CreateIndexRequest(
        parent=f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/{COLLECTION_ID}",
        index_id=f"idx-{index_field.replace('_', '-')}",
        index=index,
    )
    
    # Make the request
    return client.create_index(request=request)

In [ ]:
operation = create_index("image_embedding")
while operation.done() == False:
    time.sleep(1)
print(datetime.now())
time.sleep(30)

In [ ]:
operation = create_index("text_embedding")
while operation.done() == False:
    time.sleep(1)
print(datetime.now())